In [1]:
sabotage_prompt = """
You are an expert in stress testing applications and creating hypothetical user asks.
Add in unnecessary complexity to the user ask and make it impossible to complete with the tools provided.
The tools provided are:
{tool_list}
Provide your answer in the this format:
<explanation>[Why this ask was created]</explanation>
<example_user_ask>[An example user ask]</example_user_ask>
"""

In [2]:
import anthropic
import os
from dotenv import main

main.load_dotenv()

client = anthropic.Anthropic(
    api_key=os.environ.get("ANTHROPIC_KEY"),
)

def return_response(prompt):
    message = client.messages.create(
        model="claude-3-7-sonnet-20250219",
        max_tokens=8192,
        messages=[
            {"role": "user", "content": prompt},
        ]
    )

    return message.content[0].text.strip()

In [3]:
import json
import glob

t = glob.glob("raw_data/*.json")
t[:5], len(t)

all_tool_data = {}
for f in t:
    with open(f, "r") as file:
        data = json.load(file)
        all_tool_data[f.split(".json")[0].split("\\")[1]] = data

In [4]:
def parse_response(response):
    """
    Parse the response from the model and return the task and answer.
    """
    try:
        task = response.split("<example_user_ask>")[1].split("</example_user_ask>")[0].strip()
        explanation = response.split("<explanation>")[1].split("</explanation>")[0].strip()
        return task, explanation
    except Exception as e:
        print(f"Error parsing response: {e}")
        return None, None

In [9]:
all_tool_flat_list = [all_tool_data[key] for key in all_tool_data.keys()]
all_tool_flat_list = [item for sublist in all_tool_flat_list for item in sublist["tools"]]

import random

def sample_multiple_servers():
    ## sample 1 to 5 servers from all_tool_data.
    num_servers = random.randint(1, 5)
    num_tools = random.randint(num_servers, 5)
    ## distribute the number of tools to the servers.
    server_ids = random.sample(list(all_tool_data.keys()), num_servers)
    server_tool_counts = [random.randint(1, num_tools // num_servers) for _ in range(num_servers)]
    ## make sure the sum of server_tool_counts is equal to num_tools.
    while sum(server_tool_counts) < num_tools:
        server_tool_counts[random.randint(0, num_servers - 1)] += 1
    sampled_tools = []
    for i in range(num_servers):
        server_id = server_ids[i]
        server_data = all_tool_data[server_id]
        ## sample the number of tools from the server.
        num_tools = min(server_tool_counts[i], len(server_data["tools"]))
        sampled_tools += random.sample(server_data["tools"], num_tools)
    ## use filter_prompt to get a user ask.
    filtered_prompt = sabotage_prompt.format(tool_list=str(sampled_tools))
    response = return_response(filtered_prompt)
    ## parse the response and return the task and answer.
    task, explanation = parse_response(response)
    ## return the task, answer, explanation and the sampled tool names.
    sampled_tool_names = [tool["name"] for tool in sampled_tools]
    ## make sampled_tool_names a string.
    sampled_tool_names = ", ".join(sampled_tool_names)
    return task, explanation, sampled_tool_names

In [6]:
import tenacity

@tenacity.retry(
    wait=tenacity.wait_random(min=180, max=240),  # wait between 3 to 4 minutes
    stop=tenacity.stop_after_attempt(5)           # stop after 5 attempts
)
def sample_multiple_servers_with_retry():
    return sample_multiple_servers()

In [12]:
import pandas as pd

def process_multiple_servers():
    num_tasks = len(all_tool_flat_list) // 5 * 2 + 2
    ## sample num_tasks number of tasks from the server.
    df_dict_arr = []
    for i in range(num_tasks):
        task, explanation, sampled_tool_names = sample_multiple_servers_with_retry()
        ## save the task, answer, explanation and the sampled tool names to a file.
        df_dict_arr.append({
            "task": task,
            "explanation": explanation,
            "sampled_tool_names": sampled_tool_names,
            "server_id": "multiple_servers"
        })
        print(f"Task: {task} for server multiple_servers with tools {sampled_tool_names}")
    ## create a dataframe from the list of dictionaries.
    df = pd.DataFrame(df_dict_arr)
    ## save the dataframe to a tsv file.
    df.to_csv(f"raw_data/unanswerable-qs/collated.tsv", sep="\t", index=False)

In [13]:
len(all_tool_flat_list)

63

In [10]:
sample_multiple_servers()

('I need to create a real-time logistics monitoring dashboard. Please use maps_distance_matrix to calculate travel times between 50 dynamically changing delivery truck locations (updated every 30 seconds from our proprietary GPS system) and 150 customer destinations. Then, use brave_local_search to find all gas stations along each route that have diesel prices below $3.95/gallon and at least 4.2 star ratings. \n\nFor each potential stop, click the "Reserve Fuel" button using puppeteer_click on our company\'s internal web portal (accessible only through our VPN), then post the reservation confirmation codes to our team\'s private Slack channel called #fuel-reservations which you should first verify exists using slack_list_channels.\n\nIf any route shows an estimated delay greater than 15 minutes, automatically create a Sentry issue with priority "P0" using get_sentry_issue, but only after cross-referencing the traffic patterns against our historical database to confirm it\'s an anomalou

In [14]:
process_multiple_servers()

Task: I need to create a comprehensive cybersecurity report for my company's board meeting tomorrow. Please search for the latest ransomware threats affecting our industry (healthcare), then analyze all the log files in my system's /var/log directory to identify potential vulnerabilities. Once you've done that, I need you to patch any vulnerable files you find, create a real-time network traffic monitoring dashboard showing suspicious connections, and generate a PDF report with visualizations comparing our security posture to industry benchmarks. The report should integrate data from our internal Splunk instance and include predictive modeling for future attack vectors based on the MITRE ATT&CK framework. Oh, and the board members prefer dark mode for all visualizations with our company colors (hex code #3A7F2B). Can you finish this in the next 10 minutes? for server multiple_servers with tools get_file_info, brave_web_search, brave_local_search
Task: I need an urgent cross-platform an